# EE/CS 148B HW3: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime.
- Put the `hw3` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [ ]:
%%capture
!pip -q install -U torch torchvision numpy==1.26.0 transformers datasets "sentence-transformers<4.0" accelerate pillow tqdm matplotlib wandb pyyaml einops pytest pytest-cov
!pip -q install ninja packaging
!pip -q install flash-attn --no-build-isolation

In [ ]:
!pip -q install numpy==1.26.0 --force-reinstall

In [ ]:
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw3/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw3-sols')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


In [ ]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

In [ ]:
import gc
import math
import random
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

import basics
import vlm
from basics.lora import LoRALinear, apply_lora_to_attention
from basics.model import Block, MultiHeadAttention
from basics.rope import RoPE1D, RoPE2D
from basics.text_encoder import FrozenTextEncoder
from basics.vit import PatchEmbeddings, ViT
from vlm.clip import ProjectionHeads, clip_loss, init_logit_scale
from vlm.data import (
    EUROSAT_CLASSES,
    build_clevr_loaders,
    build_eurosat_loaders,
    build_resisc45_loaders,
)
from vlm.eval import batch_clevr_accuracy, clevr_exact_match, zeroshot_classification_accuracy
from vlm.masking import build_image_bidir_mask
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector

SEED = 0
set_seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass

CONFIGS = {
    'clip': REPO_ROOT / 'configs' / 'clip_eurosat.yaml',
    'lora': REPO_ROOT / 'configs' / 'lora_resisc.yaml',
    'vlm': REPO_ROOT / 'configs' / 'vlm_clevr.yaml',
}
for name, path in CONFIGS.items():
    assert path.exists(), f'Missing {name} config: {path}'
    with open(path) as f:
        _ = yaml.safe_load(f)

print('HW3 imports and configs loaded.')


## Your HW3 code starts here